In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import optuna
from sklearn.model_selection import KFold, StratifiedKFold

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import lightgbm as lgb

ROOT = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / "data" / "processed"

X = pd.read_parquet(DATA_PROCESSED / "X.parquet")
Y = pd.read_parquet(DATA_PROCESSED / "Y.parquet")


In [28]:
# ============================================================
# Copie du dataset
# ============================================================

X_physical_new = X.copy()


# ============================================================
# 1) Indice enveloppe thermique (isolation)
# ============================================================

R_cols = [
    "in.insulation_ceiling",
    "in.insulation_wall",
    "in.insulation_roof",
    "in.insulation_floor",
    "in.insulation_foundation_wall",
    "in.insulation_slab"
]

R_weights = {
    "in.insulation_ceiling": 0.30,
    "in.insulation_roof": 0.25,
    "in.insulation_wall": 0.25,
    "in.insulation_floor": 0.10,
    "in.insulation_foundation_wall": 0.05,
    "in.insulation_slab": 0.05
}


#X_physical_new["in.thermal_envelope_index"] = sum(X_physical_new[c].fillna(0) * w for c, w in R_weights.items())



R = X_physical_new[R_cols].replace(0, np.nan)


n = R.notna().sum(axis=1)


X_physical_new["in.thermal_envelope_index"] = (
    n / (1 / R).sum(axis=1)
)

# ============================================================
# 2) Indice pertes thermiques global
# ============================================================
'''
X_physical_new["in.heat_loss_index"] = (
    1 /
    (X_physical_new["in.thermal_envelope_index"] + 1e-6)
    +
    X_physical_new["in.air_leakage_to_outside_ach50"]
    +
    X_physical_new["in.window_ufactor"]
)

'''

# ============================================================
# 3) Indice fenêtres / gains solaires
# ============================================================

X_physical_new["in.window_index"] = (
    X_physical_new["in.window_shgc"]
    *
    X_physical_new["in.window_front"]
) / (
    X_physical_new["in.window_ufactor"] + 1e-6
)



X_physical_new["in.solar_gain_index"] = (
    X_physical_new["in.window_shgc"]
    *
    X_physical_new["in.window_front"]
    *
    (
        1 +
        X_physical_new["in.orientation_cos"]
    )
)



# ============================================================
# 4) Orientation cyclique
# ============================================================

X_physical_new["in.orientation"] = np.arctan2(
    X_physical_new["in.orientation_sin"],
    X_physical_new["in.orientation_cos"]
)





# ============================================================
# 5) Complexité bâtiment
# ============================================================




# ============================================================
# 6) Exposition thermique voisinage
# ============================================================

X_physical_new["in.thermal_exposure"] = (
    X_physical_new["in.neighbor_distance_ft"]
    -
    10 * X_physical_new["in.neighbor_both_sides"]
    +
    5 * X_physical_new["in.horiz_loc_Right"]
    +
    5 * X_physical_new["in.horiz_loc_Middle"]
)



# ============================================================
# 7) HVAC performance
# ============================================================
'''
X_physical_new["in.hvac_efficiency_index"] = (
    X_physical_new["in.hvac_cooling_efficiency"]
    *
    X_physical_new["in.hvac_cooling_partial_space_conditioning"]
    *
    (1 - X_physical_new["in.duct_leakage"])
)

'''

# ============================================================
# 8) Flexibilité thermostat
# ============================================================

X_physical_new["in.thermostat_flexibility"] = (
    X_physical_new["in.cooling_setpoint_has_offset"]
    +
    X_physical_new["in.heating_setpoint_has_offset"]
)



X_physical_new["in.setpoint_shift_capacity"] = (
    abs(
        X_physical_new[
            "in.cooling_setpoint_offset_magnitude"
        ]
    )
    +
    abs(
        X_physical_new[
            "in.heating_setpoint_offset_magnitude"
        ]
    )
)



# ============================================================
# 9) Occupation
# ============================================================

X_physical_new["in.occupancy_intensity"] = (
    X_physical_new["in.occupants"]
    /
    (
        X_physical_new["in.geometry_floor_area"]
    )
)



# ============================================================
# 10) Eau chaude sanitaire
# ============================================================
# On peut ajouter 'in.water_heater_efficiency'
X_physical_new["in.dhw_intensity"] = (
    X_physical_new["in.hot_water_fixtures"]
    *
    X_physical_new["in.occupants"]
)




# ============================================================
# Suppression des variables utilisées dans les features fusionnées
# ============================================================

drop_cols = [

    # =====================================================
    # Isolation -> remplacée par thermal_envelope_index
    # =====================================================
    "in.insulation_ceiling",
    "in.insulation_wall",
    "in.insulation_roof",
    "in.insulation_floor",
    "in.insulation_foundation_wall",
    "in.insulation_slab",


    # =====================================================
    # Fenêtres -> remplacées par window_index + solar_gain_index
    # =====================================================
    "in.window_shgc",
    "in.window_front",
    "in.window_ufactor",


    # =====================================================
    # Orientation -> remplacée par orientation
    # =====================================================
    "in.orientation_sin",
    "in.orientation_cos",


    # =====================================================
    # Géométrie -> remplacée par building_complexity
    # =====================================================



    # =====================================================
    # Voisinage -> remplacé par thermal_exposure
    # =====================================================
    "in.neighbor_distance_ft",
    "in.neighbor_both_sides",
    "in.horiz_loc_Right",
    "in.horiz_loc_Middle",


    # =====================================================
    # HVAC -> remplacé par hvac_efficiency_index
    # =====================================================
    "in.hvac_cooling_efficiency",
    "in.hvac_cooling_partial_space_conditioning",
    "in.duct_leakage",


    # =====================================================
    # Thermostat -> remplacé par flexibilité thermostat
    # =====================================================
    "in.cooling_setpoint_has_offset",
    "in.heating_setpoint_has_offset",
    "in.cooling_setpoint_offset_magnitude",
    "in.heating_setpoint_offset_magnitude",


    # =====================================================
    # Occupation -> remplacé par occupancy_intensity
    # =====================================================
    "in.occupants",
    "in.geometry_floor_area",


    # =====================================================
    # Eau chaude sanitaire -> remplacé par dhw_intensity
    # =====================================================
    "in.hot_water_fixtures",

    # =====================================================
    # Finitions murs (faible valeur physique après fusion)
    # =====================================================
    "in.wall_finish_dark",
    "in.wall_finish_brick",
    "in.wall_finish_none",
    "in.wall_finish_shingle",
    "in.wall_finish_stucco",
    "in.wall_finish_vinyl",
    "in.wall_finish_wood",
]


X_physical_new = X_physical_new.drop(
    columns=drop_cols,
    errors="ignore"
)



# ============================================================
# Nettoyage final
# ============================================================

# Remplacer infinis
X_physical_new = X_physical_new.replace(
    [np.inf, -np.inf],
    np.nan
)


# Remplacer NaN numériques
X_physical_new = X_physical_new.fillna(0)



# ============================================================
# Résultat
# ============================================================

print("="*60)
print("Réduction des variables")
print("="*60)

print(
    f"Ancien nombre de variables : {X.shape[1]}"
)

print(
    f"Nouveau nombre de variables : {X_physical_new.shape[1]}"
)

print(
    f"Variables supprimées : {X.shape[1]-X_physical_new.shape[1]}"
)



# ============================================================
# Sauvegarde
# ============================================================

X_physical_new.to_parquet(DATA_PROCESSED / 'X_physical_engineered.parquet', index=False)

print(f'X_physical_engineered sauvegardé : {DATA_PROCESSED}/X_physical_engineered.parquet  {X_physical_new.shape}')

Réduction des variables
Ancien nombre de variables : 107
Nouveau nombre de variables : 84
Variables supprimées : 23
X_physical_engineered sauvegardé : \\FS-SOP\Staff-CMA\yzouarhi\Bureau\Data\FlexiMax\data\processed/X_physical_engineered.parquet  (549971, 84)
